# Phenotype preprocessing

Prepare molecular-phenotype matrices for xQTL analysis by imputing missing values when needed, adding genomic annotations, and formatting data for chromosome-, region-, TAD-, or sample-specific analyses.


#### Miniprotocol Timing

This is the total duration for the standard toy-data route; module-specific timings appear on their respective pages.

Timing: <12 min (on the toy dataset)


## Overview

This mini-protocol walks through how molecular-phenotype matrices are prepared for downstream xQTL analysis. Each step calls a workflow from [`phenotype_imputation.ipynb`](https://statfungen.github.io/xqtl-protocol/code/data_preprocessing/phenotype/phenotype_imputation.html), [`gene_annotation.ipynb`](https://statfungen.github.io/xqtl-protocol/code/data_preprocessing/phenotype/gene_annotation.html), or [`phenotype_formatting.ipynb`](https://statfungen.github.io/xqtl-protocol/code/data_preprocessing/phenotype/phenotype_formatting.html).

The commands are selectable routes rather than one mandatory chain. Impute only matrices containing missing values. Choose the annotation workflow matching gene, protein, LeafCutter, or psichomics identifiers, then choose the formatting workflow required by the downstream analysis unit. GCT sample extraction and BAM subsetting are independent utilities.


## Steps

Choose a route before running commands; the 12 commands are not one mandatory chain.

| **Analysis goal** | **Commands to run, in order** | **Inputs** |
| --- | --- | --- |
| Gene-expression or protein phenotype by chromosome | 2 → 7 | `tests/fixtures/gene_annotation/protocol_example.rnaseq.bed.gz`; `input/reference_data/Homo_sapiens.GRCh38.103.chr.reformatted.collapse_only.gene.ERCC.gtf` |
| Phenotype with missing values by chromosome | 1 → 2 → 7 | `tests/fixtures/phenotype_imputation/protocol_example.protein.missing.bed.gz`; gene-coordinate GTF above |
| Retrieve gene coordinates from Ensembl BioMart | 3 | `tests/fixtures/gene_annotation/protocol_example.rnaseq.gene_ID.tsv`; internet access |
| LeafCutter clusters mapped to genes | 4 | `tests/fixtures/gene_annotation/protocol_example.leafcutter.phenotype.bed.gz`; `tests/fixtures/gene_annotation/protocol_example.leafcutter.intron_count.tsv`; `input/reference_data/Homo_sapiens.GRCh38.103.chr.gtf` |
| LeafCutter isoforms annotated for QTL analysis | 5 | `tests/fixtures/gene_annotation/protocol_example.leafcutter.phenotype.bed.gz`; `tests/fixtures/gene_annotation/protocol_example.leafcutter.intron_count.tsv`; `input/reference_data/Homo_sapiens.GRCh38.103.chr.gtf` |
| psichomics isoforms annotated for QTL analysis | 6 | `tests/fixtures/gene_annotation/protocol_example.psichomics.phenotype.tsv`; `input/reference_data/Homo_sapiens.GRCh38.103.chr.gtf` |
| Partition a GCT matrix by chromosome | 8 | `input/rnaseq/protocol_example.rnaseq.gene_tpm.gct.gz` |
| Partition a BED phenotype by predefined regions | 2 → 9 | `tests/fixtures/gene_annotation/protocol_example.rnaseq.bed.gz`; gene-coordinate GTF above; `input/reference_data/TAD/protocol_example_protein.enhanced_cis_chr22.bed` |
| Define TAD-based phenotype regions | 2 → 10 | `tests/fixtures/gene_annotation/protocol_example.rnaseq.bed.gz`; gene-coordinate GTF above; `tests/fixtures/generalized_TADB/expected/TADB_enhanced_cis.bed` |
| Extract selected samples from a GCT matrix | 11 | `input/rnaseq/protocol_example.rnaseq.gene_tpm.gct.gz`; `tests/fixtures/phenotype_formatting/keep_samples.txt` |
| Subset BAM files to selected genomic regions | 12 | `input/rnaseq/bam_file_list.txt`; referenced BAM files; no example BAM is currently bundled |

Run only the row matching the intended analysis goal, following its commands in numerical order. Other imputation methods are available through `phenotype_imputation.ipynb` and its Command Interface.

### 1. [Impute missing phenotype values](https://statfungen.github.io/xqtl-protocol/code/data_preprocessing/phenotype/phenotype_imputation.html)

**What it does:** Use generalized empirical Bayes matrix factorization to complete the example protein matrix.


In [ ]:
sos run pipeline/phenotype_imputation.ipynb gEBMF \
    --phenoFile tests/fixtures/phenotype_imputation/protocol_example.protein.missing.bed.gz \
    --cwd output/phenotype_imputation_uf \
    --num_factor 30


### 2. [Add genomic coordinates to gene or protein phenotypes](https://statfungen.github.io/xqtl-protocol/code/data_preprocessing/phenotype/gene_annotation.html)

**What it does:** Join phenotype identifiers to a supplied coordinate annotation and write a coordinate-aware BED matrix.


In [ ]:
sos run pipeline/gene_annotation.ipynb annotate_coord \
    --cwd output/gene_annotation \
    --phenoFile tests/fixtures/gene_annotation/protocol_example.rnaseq.bed.gz \
    --coordinate-annotation tests/fixtures/gene_annotation/Homo_sapiens.GRCh38.103.collapse_only.gene.chr22.gtf.gz \
    --phenotype-id-column gene_id


### 3. [Retrieve gene coordinates from Ensembl BioMart](https://statfungen.github.io/xqtl-protocol/code/data_preprocessing/phenotype/gene_annotation.html)

**What it does:** Query the selected Ensembl release when a local coordinate annotation is unavailable.


In [ ]:
sos run pipeline/gene_annotation.ipynb annotate_coord_biomart \
    --cwd output/gene_annotation \
    --phenoFile tests/fixtures/gene_annotation/protocol_example.rnaseq.gene_ID.tsv \
    --ensembl-version 115


### 4. [Map LeafCutter clusters to genes](https://statfungen.github.io/xqtl-protocol/code/data_preprocessing/phenotype/gene_annotation.html)

**What it does:** Assign LeafCutter clusters to genes using splice-site overlap with the gene annotation.


In [ ]:
sos run pipeline/gene_annotation.ipynb map_leafcutter_cluster_to_gene \
    --cwd output/gene_annotation \
    --phenoFile tests/fixtures/gene_annotation/protocol_example.leafcutter.phenotype.bed.gz \
    --intron-count tests/fixtures/gene_annotation/protocol_example.leafcutter.intron_count.tsv \
    --coordinate-annotation <path/to/Homo_sapiens.GRCh38.103.chr.gtf> \
    --map-stra site


### 5. [Annotate LeafCutter isoforms](https://statfungen.github.io/xqtl-protocol/code/data_preprocessing/phenotype/gene_annotation.html)

**What it does:** Convert LeafCutter intron-cluster phenotypes into annotated isoform features for QTL analysis.


In [ ]:
sos run pipeline/gene_annotation.ipynb annotate_leafcutter_isoforms \
    --cwd output/gene_annotation \
    --phenoFile tests/fixtures/gene_annotation/protocol_example.leafcutter.phenotype.bed.gz \
    --intron-count tests/fixtures/gene_annotation/protocol_example.leafcutter.intron_count.tsv \
    --coordinate-annotation <path/to/Homo_sapiens.GRCh38.103.chr.gtf> \
    --map-stra site


### 6. [Annotate psichomics isoforms](https://statfungen.github.io/xqtl-protocol/code/data_preprocessing/phenotype/gene_annotation.html)

**What it does:** Add genomic and gene annotations to psichomics-derived splicing phenotypes.


In [ ]:
sos run pipeline/gene_annotation.ipynb annotate_psichomics_isoforms \
    --cwd output/gene_annotation \
    --phenoFile tests/fixtures/gene_annotation/protocol_example.psichomics.phenotype.tsv \
    --coordinate-annotation <path/to/Homo_sapiens.GRCh38.103.chr.gtf>


### 7. [Partition a BED phenotype by chromosome](https://statfungen.github.io/xqtl-protocol/code/data_preprocessing/phenotype/phenotype_formatting.html)

**What it does:** Split a coordinate-annotated BED phenotype into chromosome-specific files.


In [ ]:
sos run pipeline/phenotype_formatting.ipynb phenotype_by_chrom \
    --cwd output/phenotype_uf \
    --phenoFile tests/fixtures/phenotype_formatting/protocol_example.rnaseq.bed.bed.gz \
    --name protocol_example \
    --chrom chr22


### 8. [Partition a GCT phenotype by chromosome](https://statfungen.github.io/xqtl-protocol/code/data_preprocessing/phenotype/phenotype_formatting.html)

**What it does:** Split a coordinate-aware GCT matrix into chromosome-specific GCT files.


In [ ]:
sos run pipeline/phenotype_formatting.ipynb phenotype_by_chrom_gct \
    --cwd output/phenotype_gct \
    --phenoFile output/phenotype/phenotype_by_chrom_for_cis/protocol_example.rnaseq.gene_tpm.gct.gz \
    --chrom chr21 chr22


### 9. [Partition a phenotype by predefined regions](https://statfungen.github.io/xqtl-protocol/code/data_preprocessing/phenotype/phenotype_formatting.html)

**What it does:** Extract phenotype features falling within each region in a supplied region list.


In [ ]:
sos run pipeline/phenotype_formatting.ipynb phenotype_by_region \
    --cwd output/phenotype_by_region \
    --phenoFile tests/fixtures/phenotype_formatting/protocol_example.rnaseq.bed.bed.gz \
    --region-list output/phenotype/phenotype_by_chrom_for_cis/protocol_example_protein.enhanced_cis_chr22.bed


### 10. [Define TAD-based phenotype regions](https://statfungen.github.io/xqtl-protocol/code/data_preprocessing/phenotype/phenotype_formatting.html)

**What it does:** Assign phenotype features to TAD windows and generate a region list for downstream analysis.


In [ ]:
sos run pipeline/phenotype_formatting.ipynb phenotype_annotate_by_tad \
    --cwd output/phenotype_by_region \
    --phenoFile tests/fixtures/phenotype_formatting/protocol_example.rnaseq.bed.bed.gz \
    --TAD-list tests/fixtures/generalized_TADB/expected/TADB_enhanced_cis.bed \
    --phenotype-per-tad 2


### 11. [Extract selected samples from a GCT matrix](https://statfungen.github.io/xqtl-protocol/code/data_preprocessing/phenotype/phenotype_formatting.html)

**What it does:** Retain only samples listed in a supplied keep file.


In [ ]:
sos run pipeline/phenotype_formatting.ipynb gct_extract_samples \
    --cwd output/phenotype_gct \
    --phenoFile output/phenotype/phenotype_by_chrom_for_cis/protocol_example.rnaseq.gene_tpm.gct.gz \
    --keep-samples tests/fixtures/phenotype_formatting/keep_samples.txt


### 12. [Subset BAM files by genomic region](https://statfungen.github.io/xqtl-protocol/code/data_preprocessing/phenotype/phenotype_formatting.html)

**What it does:** Extract selected chromosomes or regions from every BAM listed in the input manifest.


In [ ]:
sos run pipeline/phenotype_formatting.ipynb bam_subsetting \
    --cwd output/bam_subset \
    --phenoFile output/phenotype/phenotype_by_chrom_for_cis/bam_file_list.txt \
    --region chr21 chr22


## Output Files

| Route or step | Products and relative paths |
| --- | --- |
| Step 1 | `output/phenotype_imputation_uf/protocol_example.protein.missing.bed.imputed.bed.gz` |
| Step 2 | `tests/fixtures/phenotype_formatting/protocol_example.rnaseq.bed.bed.gz`; `tests/fixtures/gene_annotation/expected/protocol_example.rnaseq.bed.region_list.txt` |
| Step 3 | `output/gene_annotation/protocol_example.rnaseq.gene_ID.bed.gz` |
| Step 4 | `output/gene_annotation/protocol_example.leafcutter.phenotype.bed.gz.exon_list`; `output/gene_annotation/protocol_example.leafcutter.phenotype.bed.gz.leafcutter.clusters_to_genes.txt` |
| Step 5 | `tests/fixtures/gene_annotation/expected/protocol_example.leafcutter.phenotype.bed.formated.bed.gz`; `output/gene_annotation/protocol_example.leafcutter.phenotype.phenotype_group.txt` |
| Step 6 | `tests/fixtures/gene_annotation/expected/protocol_example.psichomics.phenotype.formated.bed.gz`; `tests/fixtures/gene_annotation/expected/protocol_example.psichomics.phenotype.phenotype_group.txt` |
| Step 7 | `output/phenotype_uf/protocol_example.genotype.chr22.bed.gz`; `tests/fixtures/phenotype_formatting/expected/protocol_example.phenotype_by_chrom_files.txt`; its region list |
| Step 8 | `output/phenotype_gct/protocol_example.rnaseq.gene_tpm.chr21.gct`; `output/phenotype_gct/protocol_example.rnaseq.gene_tpm.chr22.gct` |
| Step 9 | `output/phenotype_by_region/protocol_example_protein.enhanced_cis_chr22_phenotype_by_region/*.bed.gz`; `output/phenotype_by_region/*.phenotype_by_region_files.txt` |
| Step 10 | `output/phenotype_by_region/*_pheno_per_region.region_list`; TAD-annotated region list under `output/phenotype_by_region/` |
| Step 11 | `output/phenotype_gct/protocol_example.rnaseq.gene_tpm.sample_matched.gct.gz` |
| Step 12 | `output/bam_subset/*.subsetted.bam` |

## Anticipated Results

The selected route produces a molecular-phenotype matrix with the genomic annotation and layout required by its downstream xQTL analysis. Standard gene-expression routes typically end with chromosome-specific BED files; splicing routes end with gene- or isoform-annotated phenotypes; region and TAD routes end with per-region files and their manifest.

Continue with covariate preprocessing and then the appropriate association-testing workflow.


## Command interface

List the workflows and parameters available in each module used by this mini-protocol.


In [ ]:
sos run pipeline/phenotype_imputation.ipynb -h
sos run pipeline/gene_annotation.ipynb -h
sos run pipeline/phenotype_formatting.ipynb -h
